In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\lekshmi\Downloads\gold_lstm_features_engineered.csv")
df['Date'] = pd.to_datetime(df['Date'])

print("Earliest:", df['Date'].min())
print("Latest:  ", df['Date'].max())
print("Total rows:", len(df))
print("Columns:", df.columns.tolist())

Earliest: 2018-01-30 00:00:00
Latest:   2025-12-12 00:00:00
Total rows: 2016
Columns: ['Date', 'Gold_Close', 'USD_Close', 'avg_sentiment', 'news_count', 'Gold_7day_MA', 'Gold_30day_MA', 'USD_7day_MA']


In [3]:
pip install yfinance

  Using cached yfinance-1.2.0-py2.py3-none-any.whl.metadata (6.1 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached multitasking-0.0.12-py3-none-any.whl
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached frozendict-2.4.7-py3-none-any.whl.metadata (23 kB)
  Using cached peewee-4.0.2-py3-none-any.whl.metadata (8.6 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached curl_cffi-0.13.0-cp39-abi3-win_amd64.whl.metadata (13 kB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached websockets-16.0-cp311-cp311-win_amd64.whl.metadata (7.0 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached cffi-2.0.0-cp311-cp311-win_amd64.whl.metadata (2.6 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached charset_normalizer-3.4.6-cp311-cp311-win_amd64.whl.metadata (41 kB)
  Using cached idna-3.11-py3-


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import requests
import xml.etree.ElementTree as ET
import time

RSS_URLS = [
    "https://finance.yahoo.com/rss/topstories",
    "https://www.cnbc.com/id/100003114/device/rss/rss.html",
    "https://finance.yahoo.com/news/rssindex",
]

gold_keywords = ["gold", "bullion", "XAU", "fed", "federal reserve",
                 "interest rate", "inflation", "CPI", "USD", "dollar",
                 "recession", "commodity", "safe haven", "rate hike", "rate cut"]

def get_wayback_rss(url, year, month):
    timestamp = f"{year}{month:02d}15120000"
    archived_url = f"https://web.archive.org/web/{timestamp}/{url}"
    try:
        response = requests.get(archived_url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
        root = ET.fromstring(response.content)
        articles = []
        for item in root.findall(".//item"):
            title = item.findtext("title", "")
            if any(kw.lower() in title.lower() for kw in gold_keywords):
                articles.append({
                    "title": title,
                    "source": url.split("/")[2],
                    "published_at": item.findtext("pubDate", ""),
                    "url": item.findtext("link", ""),
                    "description": item.findtext("description", "")
                })
        return articles
    except Exception as e:
        print(f"  Error: {e}")
        return []

# Fetch Dec 2025 to Mar 2026
targets = [
    (2025, 12),
    (2026, 1), (2026, 2), (2026, 3)
]

extra = []
for year, month in targets:
    for rss_url in RSS_URLS:
        print(f"Fetching {year}-{month:02d} from {rss_url.split('/')[2]}...")
        articles = get_wayback_rss(rss_url, year, month)
        print(f"  → {len(articles)} articles")
        extra.extend(articles)
        time.sleep(2)

print(f"\nTotal new articles: {len(extra)}")

Fetching 2025-12 from finance.yahoo.com...
  → 5 articles
Fetching 2025-12 from www.cnbc.com...
  → 2 articles
Fetching 2025-12 from finance.yahoo.com...
  → 0 articles
Fetching 2026-01 from finance.yahoo.com...
  → 5 articles
Fetching 2026-01 from www.cnbc.com...
  → 0 articles
Fetching 2026-01 from finance.yahoo.com...
  → 0 articles
Fetching 2026-02 from finance.yahoo.com...
  → 5 articles
Fetching 2026-02 from www.cnbc.com...
  Error: mismatched tag: line 21, column 2
  → 0 articles
Fetching 2026-02 from finance.yahoo.com...
  → 0 articles
Fetching 2026-03 from finance.yahoo.com...
  → 5 articles
Fetching 2026-03 from www.cnbc.com...
  Error: mismatched tag: line 21, column 2
  → 0 articles
Fetching 2026-03 from finance.yahoo.com...
  → 0 articles

Total new articles: 22


In [7]:
pip install vaderSentiment

   ---------------------------------------- 0.0/126.0 kB ? eta -:--:--
   ------ -------------------------------- 20.5/126.0 kB 640.0 kB/s eta 0:00:01
   ---------------------------------------  122.9/126.0 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 126.0/126.0 kB 1.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import requests
import xml.etree.ElementTree as ET
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

# Fetch live RSS for recent headlines
LIVE_RSS = [
    "https://finance.yahoo.com/rss/topstories",
    "https://finance.yahoo.com/news/rssindex",
    "https://feeds.content.dowjones.io/public/rss/mw_realtimeheadlines",
]

gold_keywords = ["gold", "bullion", "XAU", "fed", "federal reserve",
                 "interest rate", "inflation", "CPI", "USD", "dollar",
                 "recession", "commodity", "safe haven", "rate hike", "rate cut"]

live_articles = []
for url in LIVE_RSS:
    try:
        response = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
        root = ET.fromstring(response.content)
        for item in root.findall(".//item"):
            title = item.findtext("title", "")
            if any(kw.lower() in title.lower() for kw in gold_keywords):
                live_articles.append({
                    "title": title,
                    "source": url.split("/")[2],
                    "published_at": item.findtext("pubDate", ""),
                    "url": item.findtext("link", ""),
                    "description": item.findtext("description", "")
                })
        print(f"✅ {url.split('/')[2]}: {len(live_articles)} articles")
    except Exception as e:
        print(f"Error {url}: {e}")

# Combine wayback + live
all_new = extra + live_articles
print(f"\nWayback articles: {len(extra)}")
print(f"Live articles: {len(live_articles)}")
print(f"Total: {len(all_new)}")

✅ finance.yahoo.com: 6 articles
✅ finance.yahoo.com: 12 articles
✅ feeds.content.dowjones.io: 16 articles

Wayback articles: 22
Live articles: 16
Total: 38


In [11]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

analyzer = SentimentIntensityAnalyzer()

new_df = pd.DataFrame(all_new)
new_df["published_at"] = pd.to_datetime(new_df["published_at"], errors="coerce", utc=True).dt.tz_localize(None)
new_df = new_df.dropna(subset=["title", "published_at"])
new_df = new_df[new_df["published_at"] >= "2025-12-13"]
print(f"Articles after 2025-12-13: {len(new_df)}")

new_df["sentiment_score"] = new_df["title"].apply(
    lambda x: analyzer.polarity_scores(x)["compound"]
)

daily_new = new_df.groupby(new_df["published_at"].dt.date).agg(
    avg_sentiment=("sentiment_score", "mean"),
    news_count=("sentiment_score", "count")
).reset_index()
daily_new.columns = ["date", "avg_sentiment", "news_count"]
daily_new["date"] = pd.to_datetime(daily_new["date"])

print(f"New sentiment days: {len(daily_new)}")
print(daily_new)

# Load existing sentiment
old_sentiment = pd.read_csv(r"C:\Users\lekshmi\Downloads\financial\daily_sentiment.csv")
old_sentiment["date"] = pd.to_datetime(old_sentiment["date"])

# Merge
final_sentiment = pd.concat([old_sentiment, daily_new], ignore_index=True)
final_sentiment = final_sentiment.drop_duplicates(subset=["date"])
final_sentiment = final_sentiment.sort_values("date")

print(f"\nDate range: {final_sentiment['date'].min()} to {final_sentiment['date'].max()}")
print(f"Total days: {len(final_sentiment)}")

final_sentiment.to_csv(r"C:\Users\lekshmi\Downloads\financial\daily_sentiment.csv", index=False)
print("✅ Saved updated daily_sentiment.csv")

Articles after 2025-12-13: 24
New sentiment days: 6
        date  avg_sentiment  news_count
0 2025-12-14       0.077200           1
1 2025-12-15       0.229400           2
2 2026-01-05       0.199900           9
3 2026-03-03       0.000000           2
4 2026-03-23       0.210100           4
5 2026-03-24      -0.308033           6

Date range: 0001-04-18 00:00:00 to 2026-03-24 00:00:00
Total days: 935
✅ Saved updated daily_sentiment.csv


In [10]:
import os

for root, dirs, files in os.walk(r"C:\Users\lekshmi"):
    for file in files:
        if "sentiment" in file.lower():
            print(os.path.join(root, file))

C:\Users\lekshmi\.vscode\extensions\ms-python.vscode-pylance-2026.1.1\dist\.cache\local_indices\931902782f5d97768ff4b8403aacb3ccaf3b00b4c46ad750fce842734706428b\vaderSentiment.json
C:\Users\lekshmi\.vscode\extensions\ms-python.vscode-pylance-2026.1.1\dist\.cache\local_indices\bfab9145bd7d549ed15a25e198edbe834475e0a3d0ab3eafa78529cf2e32ef98\vaderSentiment.json
C:\Users\lekshmi\AppData\Local\Programs\Python\Python311\Lib\site-packages\vaderSentiment\vaderSentiment.py
C:\Users\lekshmi\AppData\Local\Programs\Python\Python311\Lib\site-packages\vaderSentiment\__pycache__\vaderSentiment.cpython-311.pyc
C:\Users\lekshmi\AppData\Local\Programs\Python\Python314\Lib\site-packages\nltk\sentiment\sentiment_analyzer.py
C:\Users\lekshmi\AppData\Local\Programs\Python\Python314\Lib\site-packages\nltk\sentiment\__pycache__\sentiment_analyzer.cpython-314.pyc
C:\Users\lekshmi\AppData\Local\Programs\Python\Python314\Lib\site-packages\nltk\test\sentiment.doctest
C:\Users\lekshmi\AppData\Local\Programs\Pytho

In [12]:
import pandas as pd

final_sentiment = pd.read_csv(r"C:\Users\lekshmi\Downloads\financial\daily_sentiment.csv")
final_sentiment["date"] = pd.to_datetime(final_sentiment["date"], errors="coerce")

# Remove corrupted dates (keep only 2017-2026)
final_sentiment = final_sentiment[
    (final_sentiment["date"] >= "2017-01-01") &
    (final_sentiment["date"] <= "2026-12-31")
]
final_sentiment = final_sentiment.sort_values("date")

print(f"Date range: {final_sentiment['date'].min()} to {final_sentiment['date'].max()}")
print(f"Total days: {len(final_sentiment)}")
print("\nYear distribution:")
print(final_sentiment["date"].dt.year.value_counts().sort_index())

final_sentiment.to_csv(r"C:\Users\lekshmi\Downloads\financial\daily_sentiment.csv", index=False)
print("✅ Saved cleaned daily_sentiment.csv")

Date range: 2017-12-17 00:00:00 to 2026-03-24 00:00:00
Total days: 935

Year distribution:
date
2017     11
2018    353
2019    365
2020    200
2025      2
2026      4
Name: count, dtype: int64
✅ Saved cleaned daily_sentiment.csv


C:\Users\lekshmi\AppData\Local\Temp\ipykernel_31036\277674713.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  final_sentiment["date"] = pd.to_datetime(final_sentiment["date"], errors="coerce")


In [13]:
import os

for root, dirs, files in os.walk(r"C:\Users\lekshmi"):
    for file in files:
        if file == "daily_sentiment.csv":
            print(os.path.join(root, file))

C:\Users\lekshmi\Downloads\financial\daily_sentiment.csv


In [15]:
import os

for root, dirs, files in os.walk(r"C:\Users\lekshmi"):
    for file in files:
        if "gold_relevant" in file.lower() or "gold_news" in file.lower():
            print(os.path.join(root, file))

C:\Users\lekshmi\AppData\Roaming\Microsoft\Office\Recent\gold_relevant_news.LNK
C:\Users\lekshmi\AppData\Roaming\Microsoft\Windows\Recent\gold_relevant_news.lnk


In [16]:
import pandas as pd

df = pd.read_csv(r"C:\Users\lekshmi\Downloads\gold_lstm_features_engineered.csv")
df['Date'] = pd.to_datetime(df['Date'])

print("Date range:", df['Date'].min(), "to", df['Date'].max())
print("Columns:", df.columns.tolist())
print("Rows:", len(df))
print("\nSentiment stats:")
print(df['avg_sentiment'].describe())

Date range: 2018-01-30 00:00:00 to 2025-12-12 00:00:00
Columns: ['Date', 'Gold_Close', 'USD_Close', 'avg_sentiment', 'news_count', 'Gold_7day_MA', 'Gold_30day_MA', 'USD_7day_MA']
Rows: 2016

Sentiment stats:
count    2016.000000
mean        0.000142
std         0.268661
min        -0.670500
25%        -0.065125
50%         0.010894
75%         0.102700
max         0.802000
Name: avg_sentiment, dtype: float64


In [18]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# Convert new articles to DataFrame
new_df = pd.DataFrame(all_new)
new_df["published_at"] = pd.to_datetime(new_df["published_at"], errors="coerce", utc=True).dt.tz_localize(None)
new_df = new_df.dropna(subset=["title", "published_at"])
new_df = new_df[new_df["published_at"] >= "2025-12-13"]

# Run VADER
new_df["sentiment_score"] = new_df["title"].apply(
    lambda x: analyzer.polarity_scores(x)["compound"]
)

# Aggregate by day
daily_new = new_df.groupby(new_df["published_at"].dt.date).agg(
    avg_sentiment=("sentiment_score", "mean"),
    news_count=("sentiment_score", "count")
).reset_index()
daily_new.columns = ["date", "avg_sentiment", "news_count"]
daily_new["date"] = pd.to_datetime(daily_new["date"])

print(f"New sentiment days: {len(daily_new)}")
print(daily_new)

# Save separately for your teammate to merge with new gold/USD prices
daily_new.to_csv(r"C:\Users\lekshmi\Downloads\new_sentiment_2025_2026.csv", index=False)
print("\n✅ Saved new_sentiment_2025_2026.csv")


New sentiment days: 6
        date  avg_sentiment  news_count
0 2025-12-14       0.077200           1
1 2025-12-15       0.229400           2
2 2026-01-05       0.199900           9
3 2026-03-03       0.000000           2
4 2026-03-23       0.210100           4
5 2026-03-24      -0.308033           6

✅ Saved new_sentiment_2025_2026.csv
